VALIDACIÓN DEL MÉTODO DE PRONÓSTICO HIDROLÓGICO ESP

Este código utiliza el modelo de Temez y datos de precipitación y evapotranspiración del período histórico de 1990 a 2020 para generar ensambles de pronóstico hidrológico considerando cada año de la serie histórica como forzante climática.

In [1]:
#importacion

import xarray as xr
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
from TemezRegional import TemezRegional

In [2]:
# Definiciones

lead_time = 6
start_year = 1990
end_year = 2020

#vamos a empezar un conteo para registrar errores al calcular ensambles, bastante más adelante en el código:
errores = 0

In [3]:
# Cargar datos
datos_enso = pd.read_csv(
    "../Datos_entrada/nina34_anom.csv",
    header=None
)

precip = pd.read_csv(
    "../Datos_entrada/Pmedias.csv",
    header=None
)

etp = pd.read_csv(
    "../Datos_entrada/ETPmedias.csv",
    header=None
)

ad = pd.read_csv(
    "../Datos_entrada/AguaDisponible.txt",
    delim_whitespace=True,
    header=None
)

cuencas_n3 = pd.read_csv(
    "../Datos_entrada/cuenca_nivel3.csv",
    header=None
)

In [4]:
# Crear lista de cuencas n2 a partir de las cuencas n3:

codigos_n3 = cuencas_n3.iloc[0, :].to_numpy().astype(int)
codigos_n2 = np.unique((codigos_n3 // 10).astype(int))

Acá abajo vamos a processar los datos para que tengan fecha como index y cuenca n3 como nombre de las columnas

In [5]:
#Separar metadatos

meta_precip = precip.iloc[0, :]
meta_etp = etp.iloc[0, :]
# los datos reales estan a partir de la fila 1
precip_data = precip.iloc[1:, :].copy()
etp_data = etp.iloc[1:, :].copy()

# Crear fechas (datetime)
dates = pd.to_datetime(
    dict(
        year=precip_data.iloc[:, 0].astype(int),
        month=precip_data.iloc[:, 1].astype(int),
        day=15
    )
)

dates = pd.DatetimeIndex(dates)

# Quitar año y mes de las séries
precip_data = precip_data.iloc[:, 2:].copy()
etp_data = etp_data.iloc[:, 2:].copy()

# Asignar fecha (datetime) como índice
precip_data.index = dates
etp_data.index = dates
precip_data.index.name = "time"
etp_data.index.name = "time"

# Nombrar las columnas de las series con la cuenca correspondiente
precip_data.columns = codigos_n3
etp_data.columns    = codigos_n3

#agregar index a ad y transformarlo en una serie
ad_series = pd.Series(
    ad.iloc[:, 0].values,
    index=codigos_n3,
    name="n3"
)

#print(ad_series)
#print(precip_data)
#print(etp_data)
#acá tenemos las séries históricas con index fecha y nombre de columna de la cuenca correspondiente.

Vamos a procesar los datos de intensidad de ENSO

In [6]:
enso_anom = datos_enso.iloc[:, 1]
enso_anom = enso_anom.iloc[1:]
enso_anom.index = dates
enso_anom.index.name = "time"

#print(enso_anom)

Vamos a definir una función para buscar años análogos

In [8]:
###### definiciones iniciales:
ventana = 6 #la ventana de meses que se van a considerar para buscar un año análogo
k = 26 #cantidad de análogos

def buscar_años_analogos(t0, enso_anom, ventana, k, pesos_recientes=True):

    #serie_enso_tri = enso_anom.rolling(3, min_periods=3).mean() #se calcula la media movil de 3 meses para la anomalia ENSO
    serie_enso_tri = pd.to_numeric(enso_anom, errors="raise")

    tiempo_inicial = t0 - pd.DateOffset(months=ventana-1) #primer mes del período de análisis del ENSO
    
    ventana_actual = serie_enso_tri.loc[:t0].tail(ventana) #trancando la serie en la t0 guardar los ultimos meses (ventana)
    if len(ventana_actual) != ventana:
        return []

    if pesos_recientes:
        pesos = np.linspace(1, 2, ventana)  # crea secuencia entre 1 y 2 para asignar pesos a diferentes meses
    else:
        pesos = np.ones(ventana) #si pesos_recientes=False no asigna pesos
    # normalizar pesos
    pesos = pesos / pesos.sum() #normalización

    #def fase(x, umbral=0.5):
    #    if x > umbral:
    #        return 1
    #    elif x < -umbral:
    #        return -1
    #    else:
    #        return 0
    
    #fase_actual = fase(ventana_actual.iloc[-1])

    dist_total = {}

    for i in range(ventana, len(serie_enso_tri)):

        t_end = serie_enso_tri.index[i-1] #obtener el último mes de la ventana
    
        if t_end.month != t0.month: #si el último mes de la ventana no es igual al mes de emisión si saltea esta série. 
            continue
    
        ventana_analogo = serie_enso_tri.iloc[i-ventana:i]

        # evitar mismo año 
        if abs((t_end - t0).days) < 60:
            continue

        # --- fase ---
        #if fase(ventana_analogo.iloc[-1]) != fase_actual:
        #    continue

        # --- distancia ---
        diff = ventana_actual.to_numpy() - ventana_analogo.to_numpy()
        diff_pesos = abs(diff) * pesos
        diff_final = np.sum(diff_pesos)


        dist_total[t_end] = diff_final

    if len(dist_total) == 0:
        return []

    # ordenar
    analogos = sorted(dist_total, key=dist_total.get)[:k]

    return analogos

Acá empieza un loop para correr el modelo para cada cuenca n3 y guardar el caudal y los estados internos para cada paso de tiempo.

In [9]:
#Empezamos creando contenedores
QC_historica = {} #contenedor para el caudal modelado con clima hstórico
states_internal_status = {} #contendor para los estados internos para cada paso de tiempo

for cuenca_n3 in precip_data.columns:

    P   = precip_data[cuenca_n3].to_numpy() # devuelve array numérico (série de precipitacion)
    ETP = etp_data[cuenca_n3].to_numpy() # devuelve array numérico (série de evapotranspiracion)
    AD = ad_series.loc[cuenca_n3] #.loc devuelve los valores de la etiquita cuenca.n3 (el de ad para esta cuenca)

    #correr el modelo de temez con los datos recien extraidos
    fluxes, states = TemezRegional(
        AD=AD,
        P=P,
        ETP=ETP
    )

    QC_historica[cuenca_n3] = pd.Series(fluxes["QC"], index=precip_data.index) #guardar caudal en el contenedor

    states_internal_status[cuenca_n3] = { #guardar estados internos en el contenedor
        "H": pd.Series(states["H"], index=precip_data.index),
        "V": pd.Series(states["V"], index=precip_data.index)
    }

#print (QC_historica)
#print (states_internal_status)

Hasta ahora corremos el modelo con todos los datos históricos disponibles, así tenemos los estados más calientes posibles.
Para empezar a trabajar con la matriz de ensambles, antes vamos filtrar las variables climaticas para el período deseado:

In [10]:
# Filtrar período
precip_f = precip_data.loc[str(start_year):str(end_year)]
etp_f    = etp_data.loc[str(start_year):str(end_year)]
dates_f  = precip_f.index
#print(precip_cuenca_f)
#print(dates_f)

#esto acá abajo despues se utiliza para garantinzar que no se intente correr ensambles con fechas fuera de rango
fecha_min_datos = precip_f.index.min()
fecha_max_datos = precip_f.index.max()

Ahora ya podemos empezar a calcular los ensambles. Lo que queremos es correr cada secuencia climatica histórica (ensamble) para cada cuenca con los estados iniciales de cada fecha de emisión. La extructura queda así:
Para cada "fecha de emisión" (t0) dentro del rango historico (estados inciales)
    para cada cuenca nivel 3 (n3) dentro de todas las cuencas nivel 3 (AD, eto, precip)
        para cada ano disponible en la série histórica que no sea el ano de emisión (ano) correr el modelo

In [11]:
#contenedor para los resultados
results_esp_n3 = {}

#las fechas de emisión del pronóstico son el rango histórico - el lead time
fechas_emision = dates_f[:-lead_time]
#lista de años para buscar los ensambles
anos_hist = np.unique(dates_f.year)

In [12]:
#primer bucle para cada fecha de emisión
for t0 in tqdm(fechas_emision, desc="Procesando ESP"):

    #selecciona el año de t0 y localiza su index para posteriormente poder filtrar de la lista de ensambles
    y0 = t0.year
    i0 = dates_f.get_loc(t0)

    #obtiene las fechas de pronóstico del esp (el horizonte de pronóstico)
    #dates_f tiene que ser fechas ordenadas y contínuas, porque future dates se selecciona a partir del indice de dates_f
    future_dates = dates_f[i0 + 1 : i0 + 1 + lead_time]

    #creamos un contenedor para cada fecha de emisión dentro del contenedor de resultados
    results_esp_n3[t0] = {}

    #-------- Ahora vamos para el próximo nivel de loop, por cuenca n3. ---------------#
    for cuenca_n3 in precip_f.columns:

        #para cada cuenca de nivel 3 queremos tener los estados calientes
        H0 = states_internal_status[cuenca_n3]["H"].loc[t0]
        V0 = states_internal_status[cuenca_n3]["V"].loc[t0]
    
        #también filtramos precip y evapo:
        precip_cuenca_f = precip_f[cuenca_n3]
        etp_cuenca_f = etp_f[cuenca_n3]
    
        #y también ad
        ad_cuenca = ad_series.loc[cuenca_n3]

        
        # obtener análogos
        analogos = buscar_años_analogos(
            t0,
            enso_anom,
            ventana=ventana,
            k=k
        )
    
        #creamos un contenedor para cada cuenca n3 dentro del contendor de fecha de emisión
        results_esp_n3[t0][cuenca_n3] = {}
    
        #print(precip_cuenca_f)

        #-------- Ahora vamos empezar con el loop que corre el modelo para cada año climático a partir
        #de los estados calientes de cada cada cuenca n3 para cada fecha de emisión (t0). -----------#

        
        for t_analogo in analogos:
        
            ano = t_analogo.year
        
            ens_dates = [
                d + pd.DateOffset(years=(ano - y0))
                for d in future_dates
            ]
        
            if (min(ens_dates) < fecha_min_datos) or (max(ens_dates) > fecha_max_datos):
                continue
        
            try:
                P_esp = precip_cuenca_f.loc[ens_dates].to_numpy()
                ETP_esp = etp_cuenca_f.loc[ens_dates].to_numpy()
            except KeyError:
                errores += 1
                continue
        
            fluxes, states = TemezRegional(
                AD=ad_cuenca,
                P=P_esp,
                ETP=ETP_esp,
                H0=H0,
                V0=V0,
            )
        
            QC = fluxes["QC"]
        
            results_esp_n3[t0][cuenca_n3][ano] = QC
        
        #print (results_esp_n3)
                
    

Procesando ESP: 100%|██████████████████████████████████████████████████████████████| 366/366 [1:43:58<00:00, 17.04s/it]


Ahora que ya realizamos la corrida histórica del modelo para cuencas n3 y ya calculamos todos los ensambles para cuencas n3, vamos agregar los resultados a cuencas n2, empezando por la corrida histórica.

In [13]:
#contenedor
results_hist_n2 = {}

#convertir QC_historica a un dataframe
QC_hist_df = pd.DataFrame(QC_historica)
QC_hist_df.index.name = "time"
#print(QC_hist_df[101])

#filtrar fechas para el período histórico que queremos trabajar
QC_hist_df_f = QC_hist_df.loc[dates_f]
#print(QC_hist_df_f)

#loop para agregar los valores a cada cuenca n2
for cuenca_deseada in codigos_n2:
    
    #filtrar las cuencas n3 que pertenecen a la n2 del loop
    columnas_n3 = [c for c in QC_hist_df_f.columns if c // 10 == cuenca_deseada]
    #print(QC_hist_df_n2)
    
    #sumar los valores de todas las n3
    QC_hist_n2 = QC_hist_df_f[columnas_n3].sum(axis=1)
    QC_hist_n2.name = "QC"

    #organizar el dataframe para que sea compatible con el dataframe de salida del esp
    df_temp = pd.DataFrame({
        "fecha_pronostico": QC_hist_n2.index,
        "QC_hist": QC_hist_n2.values,
        "lead": 0
    })
    
    # guardar
    results_hist_n2[cuenca_deseada] = df_temp

    
#print (results_hist_n2)

ahora vamos acumular a n2 los resultados del esp. Venimos desde una estructura que es: - para cada fecha de emisión (t0) -> todas las cuencas n3 y para cada cuenca n3 -> cada ensamble calculado

In [14]:
#empezamos el loop por cuenca n2
for cuenca_deseada in codigos_n2:

    # contenedor para la cuenca del loop
    results_cuenca = {}

    #empezamos a acceder a la extructura de los datos de results_esp_n3
    for t0 in results_esp_n3:
        results_cuenca[t0] = {} #contenedor

        for cuenca_n3 in results_esp_n3[t0]:
            
            #acá ya filtramos las cuencas n3 que pertenecen a la cuenca n2 del loop
            if cuenca_n3 // 10 != cuenca_deseada:
                continue

            for ano, QC in results_esp_n3[t0][cuenca_n3].items(): #accedemos a las los leadtimes por medio de .items

                #este bucle copia los valores de cuenca n3 si es la primera del bucle, de la segunda en adelante empieza a sumar
                if ano not in results_cuenca[t0]: 
                    results_cuenca[t0][ano] = QC.copy()
                else:
                    results_cuenca[t0][ano] += QC 
    #print(results_cuenca)


    # vamos pasar estos datos a un DataFrame

    rows = [] #creamos una lista contenedor

    #ahora vamos reorganizar los resultados
    for t0, ensambles in results_cuenca.items():
        for ano, QC in ensambles.items():
            for k, q in enumerate(QC, start=1):
                rows.append({
                    "cuenca_n2": cuenca_deseada,
                    "fecha_emision": t0,
                    "ano": ano,
                    "lead": k,
                    "fecha_pronostico": t0 + pd.DateOffset(months=k),
                    "QC": q
                })

    df_esp = pd.DataFrame(rows, columns=[
        "cuenca_n2",
        "fecha_emision",
        "ano",
        "lead",
        "fecha_pronostico",
        "QC"
    ])

    #ya tenemos los datos en dataframe, seguir organizando:
    
    df_esp_wide = (
        df_esp
        .pivot( #cambiar de columnas a filas
            index=["fecha_emision", "lead", "fecha_pronostico"],
            columns="ano",
            values="QC"
        )
        .reset_index()
    )

    #nombrar los ensanbles por año y organizarlos en columnas
    df_esp_wide.columns = [
        f"ens_{int(c)}" if isinstance(c, (int, np.integer)) else c
        for c in df_esp_wide.columns
    ]

    #el dataframe final debe tener los valores de los ensambles y de qc_hist también

    df_hist_cuenca = results_hist_n2[cuenca_deseada]
    
    df_final = (
        df_esp_wide
        .merge(
            df_hist_cuenca[["fecha_pronostico", "QC_hist"]], # <--- Usar el específico
            on="fecha_pronostico",
            how="left"
        )
        .sort_values(["fecha_emision", "lead"])
        .reset_index(drop=True)
    )

    cols_base = ["fecha_emision", "lead", "fecha_pronostico", "QC_hist"] #columnas iniciales
    cols_ens = sorted(c for c in df_final.columns if c.startswith("ens_")) #columnas de los ensambles

    df_final["ens_prom"] = df_final[cols_ens].mean(axis=1) #crear una columna con el promedio de los ensambles
    df_final = df_final[cols_base + ["ens_prom"] + cols_ens] #juntar todo

    #exportar
    df_final.to_csv(
        rf"../Resultados/Tablas_de_resultados/{cuenca_deseada}_df_esp_con_historico.csv",
        index=False,
        float_format="%.3f"
    )